# 09 Food Diary Agent (Real Nutrition Database)

## Architecture - 2-layer lookup (accuracy first)

```
User input (Bulgarian text)
        ↓
GPT-4o-mini: extract food items + quantities (NO nutrition values)
        ↓
Layer 1: USDA FoodData Central API  ← government-verified, ~300K foods
        ↓ (if not found)
Layer 2: GPT-4o-mini fallback ← for truly unknown foods only
```

**Key principle:** LLM understands text and quantities - database provides nutrition facts.

**API needed:**
- USDA FoodData Central: free API key at https://fdc.nal.usda.gov/api-guide


In [13]:
import json, os
from pathlib import Path
from datetime import datetime, date
from typing import Optional
import warnings; warnings.filterwarnings('ignore')

import httpx
from openai import OpenAI
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from tenacity import retry, wait_exponential, stop_after_attempt
load_dotenv(override=True)

NOTEBOOK_DIR = Path().resolve()
BACKEND_DIR = NOTEBOOK_DIR.parent
DATA_DIR = BACKEND_DIR / 'data' / 'food_diary'
DATA_DIR.mkdir(parents=True, exist_ok=True)

openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
LLM_MODEL = os.getenv('PRIMARY_MODEL')

USDA_API_KEY = os.getenv('USDA_API_KEY', 'DEMO_KEY')  # DEMO_KEY = 30 req/hour
USDA_BASE_URL = os.getenv('USDA_BASE_URL')

# Open Food Facts - no key needed
OFF_BASE_URL = 'https://world.openfoodfacts.org/cgi/search.pl'

print('Setup complete')
print(f'USDA API key: {USDA_API_KEY[:8]}...')

Setup complete
USDA API key: rk5pucoE...


## 1. Food Item Extractor (LLM - text only, no nutrition)

In [14]:
class FoodItem(BaseModel):
    food_name_en: str = Field(description="Food name in English for database lookup")
    food_name_bg: str = Field(description="Food name in Bulgarian for display")
    quantity_g: float = Field(description="Quantity in grams")
    cooking_method: str = Field(default="", description="raw|boiled|grilled|fried|baked - affects calories")
    notes: str = Field(default="", description="Any relevant notes e.g. 'with skin', 'whole milk'")

class ExtractedFoods(BaseModel):
    items: list[FoodItem]

@retry(wait=wait_exponential(min=1, max=20), stop=stop_after_attempt(3))
def extract_food_items(user_input: str) -> list[FoodItem]:
    """
    Use GPT-4o-mini to extract structured food items from Bulgarian text.
    ONLY extracts food names and quantities - NO nutrition values.
    Nutrition comes from USDA/Open Food Facts databases.
    """
    prompt = f"""Extract all food items and quantities from this text. 

Text: {user_input}

For each food item:
- food_name_en: English name optimized for USDA database search (e.g. "chicken breast raw", "oats dry", "whole egg")
- food_name_bg: Bulgarian name for display
- quantity_g: weight in grams (estimate typical serving if not specified)
- cooking_method: how it's prepared (affects calorie density)
- notes: relevant details like fat content, brand hints

Common conversions:
- 1 яйце = 60г | 1 банан = 120г | 1 филия хляб = 30г
- 1 с.л. масло = 15г | 1 ч.л. = 5г | 1 чаша = 240мл

Reply ONLY with valid JSON."""

    response = openai_client.chat.completions.create(
        model = LLM_MODEL,
        messages = [{'role': 'user', 'content': prompt}],
        response_format = {'type': 'json_object'},
        temperature = 0.0,
        max_tokens = 600,
    )
    data = json.loads(response.choices[0].message.content)
    items = data.get('items', [])
    return [FoodItem(**item) for item in items]

# Test extraction only
print('Testing food item extraction (no nutrition yet)...')
items = extract_food_items('4 яйца бъркани с масло и 100г овесени ядки с мляко')
for item in items:
    print(f'  EN: {item.food_name_en:<35} BG: {item.food_name_bg:<20} {item.quantity_g}г  [{item.cooking_method}]')

Testing food item extraction (no nutrition yet)...


## 2. USDA FoodData Central Lookup

In [15]:
def parse_usda_nutrients(food_data: dict, quantity_g: float) -> dict:
    """
    Extract calories + macros from USDA food object.
    Handles both /foods/search and /food/{id} response formats.
    Values in USDA are per 100g — scaled to quantity_g.
    """
    CALORIE_IDS = {1008, 1062, 2047, 2048}
    NUTRIENT_MAP = {
        1003: 'protein_g',
        1004: 'fat_g',
        1005: 'carbs_g',
        2000: 'sugar_g',
        1079: 'fiber_g',
    }

    nutrients = {}

    for n in food_data.get('foodNutrients', []):
        # /foods/search: flat dict with nutrientId + value
        # /food/{id}: nested nutrient.id + amount
        nid = n.get('nutrientId') or n.get('nutrient', {}).get('id')
        val = n.get('value') or n.get('amount') or 0

        if nid in CALORIE_IDS:
            if 'calories' not in nutrients and val > 0:
                nutrients['calories'] = val
        elif nid in NUTRIENT_MAP:
            key = NUTRIENT_MAP[nid]
            if key not in nutrients:
                nutrients[key] = val

    # Carbs can be 0 (e.g. chicken) — only fallback if calories truly missing
    if 'calories' not in nutrients or nutrients['calories'] == 0:
        p = nutrients.get('protein_g', 0)
        f = nutrients.get('fat_g', 0)
        c = nutrients.get('carbs_g', 0)
        nutrients['calories'] = round(p * 4 + f * 9 + c * 4, 1)

    scale = quantity_g / 100.0
    return {k: round(v * scale, 1) for k, v in nutrients.items()}


def search_usda(food_name_en: str, quantity_g: float) -> Optional[dict]:
    """
    Search USDA FoodData Central.
    Prefers Foundation Foods > SR Legacy for accuracy.
    Returns None if not found.
    """
    try:
        params = {
            'api_key': USDA_API_KEY,
            'query': food_name_en,
            'dataType': ['Foundation', 'SR Legacy'],
            'pageSize': 5,
            'sortBy': 'dataType.keyword',
            'sortOrder': 'asc',
        }
        r = httpx.get(f'{USDA_BASE_URL}/foods/search', params=params, timeout=10)
        r.raise_for_status()
        data = r.json()
        foods = data.get('foods', [])
        if not foods:
            return None

        food = foods[0]
        ntr = parse_usda_nutrients(food, quantity_g)

        return {
            'source': 'USDA FoodData Central',
            'food_id': food.get('fdcId'),
            'food_name': food.get('description', food_name_en),
            'data_type': food.get('dataType', ''),
            'quantity_g': quantity_g,
            **ntr,
        }
    except Exception as e:
        print(f'    USDA error for "{food_name_en}": {e}')
        return None


# Test USDA
print('Testing USDA lookup...')
for query, qty in [('chicken breast raw', 200), ('oats dry', 100), ('whole egg', 60)]:
    result = search_usda(query, qty)
    if result:
        print(f'  {query:<25} {qty}г → '
              f'{result["calories"]:.0f} ккал | '
              f'P:{result["protein_g"]:.1f}г F:{result["fat_g"]:.1f}г C:{result["carbs_g"]:.1f}г '
              f'[{result["data_type"]}]')
    else:
        print(f'  {query}: NOT FOUND')


Testing USDA lookup...
  chicken breast raw        200г → 212 ккал | P:45.0г F:3.9г C:0.0г [Foundation]
  oats dry                  100г → 360 ккал | P:23.6г F:1.9г C:62.2г [Foundation]
  whole egg                 60г → 89 ккал | P:7.4г F:6.0г C:0.6г [Foundation]


## 3. LLM Fallback (for foods not in USDA)

In [16]:
@retry(wait=wait_exponential(min=1, max=20), stop=stop_after_attempt(3))
def llm_nutrition_fallback(food_name_en: str, food_name_bg: str,
                           quantity_g: float, cooking_method: str) -> dict:
    """
    Last resort: ask GPT-4o-mini for nutrition estimate.
    Used ONLY when USDA and Open Food Facts both fail.
    Always marked as 'estimated' with low confidence.
    """
    prompt = f"""Provide nutrition facts for: {food_name_en} ({cooking_method})
Quantity: {quantity_g}g

Use standard nutritional reference values. Be conservative and accurate.
Return JSON: {{"calories": float, "protein_g": float, "fat_g": float, "carbs_g": float}}
Reply ONLY with valid JSON."""

    r = openai_client.chat.completions.create(
        model=LLM_MODEL, messages=[{'role':'user','content':prompt}],
        response_format={'type':'json_object'}, temperature=0.0, max_tokens=100,
    )
    data = json.loads(r.choices[0].message.content)
    return {
        'source': 'LLM estimate (low confidence)',
        'food_name': food_name_bg,
        'quantity_g': quantity_g,
        'calories': float(data.get('calories', 0)),
        'protein_g': float(data.get('protein_g', 0)),
        'fat_g': float(data.get('fat_g', 0)),
        'carbs_g': float(data.get('carbs_g', 0)),
    }

print('LLM fallback defined (last resort only)')

LLM fallback defined (last resort only)


## 4. Main Lookup Function - 2 Layers

In [17]:
def lookup_nutrition(item: FoodItem) -> dict:
    """
    2-layer nutrition lookup:
    Layer 1: USDA FoodData Central (government-verified, ~300K foods)
    Layer 2: GPT-4o-mini estimate  (fallback for foods not in USDA)
    """
    query = item.food_name_en
    if item.cooking_method:
        query += f' {item.cooking_method}'
    if item.notes:
        query += f' {item.notes}'

    # Layer 1: USDA
    result = search_usda(query, item.quantity_g)
    if result:
        result['food_name_display'] = item.food_name_bg
        result['confidence'] = 'high'
        return result

    # Layer 2: GPT-4o-mini fallback
    print(f'  ⚠️  "{item.food_name_bg}" not in USDA - using LLM estimate')
    result = llm_nutrition_fallback(
        item.food_name_en, item.food_name_bg,
        item.quantity_g, item.cooking_method
    )
    result['food_name_display'] = item.food_name_bg
    result['confidence'] = 'low'
    return result


def parse_food_entry(user_input: str) -> list[dict]:
    """
    Full pipeline: Bulgarian text → list of nutrition entries.
    1. LLM extracts food items + quantities
    2. Each item looked up in USDA → LLM fallback
    """
    items = extract_food_items(user_input)
    results = []
    for item in items:
        nutrition = lookup_nutrition(item)
        results.append({
            'food_name': nutrition.get('food_name_display', item.food_name_bg),
            'quantity_g': item.quantity_g,
            'calories': nutrition.get('calories', 0),
            'protein_g': nutrition.get('protein_g', 0),
            'fat_g': nutrition.get('fat_g', 0),
            'carbs_g': nutrition.get('carbs_g', 0),
            'sugar_g': nutrition.get('sugar_g', 0),
            'fiber_g': nutrition.get('fiber_g', 0),
            'source': nutrition.get('source', '?'),
            'confidence': nutrition.get('confidence', 'low'),
            'logged_at': datetime.now().isoformat(),
        })
    return results


# Full pipeline test
print('Testing full 2-layer pipeline...\n')
test_inputs = [
    '4 яйца бъркани с масло',
    '200г пилешки гърди на скара',
    '100г овесени ядки',
    'протеинов шейк с 200мл прясно мляко и 40г протеин на прах',
]
for inp in test_inputs:
    print(f'Input: {inp}')
    entries = parse_food_entry(inp)
    for e in entries:
        icon = '🟢' if e['confidence'] == 'high' else '🔴'
        print(f'  {icon} {e["food_name"]:<28} {e["quantity_g"]:>5}г | '
              f'{e["calories"]:>6.0f} ккал | '
              f'P:{e["protein_g"]:>5.1f}г F:{e["fat_g"]:>5.1f}г C:{e["carbs_g"]:>5.1f}г '
              f'[{e["source"][:25]}]')
    print()


Testing full 2-layer pipeline...

Input: 4 яйца бъркани с масло

Input: 200г пилешки гърди на скара

Input: 100г овесени ядки

Input: протеинов шейк с 200мл прясно мляко и 40г протеин на прах



## 5. Daily Log & Tracking

In [18]:
class DailyLog:
    def __init__(self, user_id: str, log_date: str = None):
        self.user_id = user_id
        self.log_date = log_date or date.today().isoformat()
        self.entries = []
        self.path = DATA_DIR / f'{user_id}_{self.log_date}.json'
        self._load()

    def _load(self):
        if self.path.exists():
            data = json.loads(self.path.read_text(encoding='utf-8'))
            self.entries = data.get('entries', [])

    def save(self):
        self.path.write_text(
            json.dumps({'user_id':self.user_id,'date':self.log_date,'entries':self.entries},
                       ensure_ascii=False, indent=2), encoding='utf-8')

    def add(self, entries: list[dict]):
        self.entries.extend(entries)
        self.save()

    def totals(self) -> dict:
        return {k: round(sum(e.get(k,0) for e in self.entries), 1)
                for k in ['calories','protein_g','fat_g','carbs_g']}

    def clear(self):
        self.entries = []
        self.save()


def log_food(user_id: str, text: str, log_date: str = None) -> dict:
    diary = DailyLog(user_id, log_date)
    entries = parse_food_entry(text)
    diary.add(entries)
    return {'entries_added': entries, 'daily_totals': diary.totals()}


def get_daily_summary(user_id: str, targets: dict, log_date: str = None) -> dict:
    diary = DailyLog(user_id, log_date)
    totals = diary.totals()
    remaining = {k: round(targets.get(k,0) - totals.get(k,0), 1)
                 for k in ['calories','protein_g','fat_g','carbs_g']}
    pct = {k: round(totals.get(k,0) / targets.get(k,1) * 100, 1)
                 for k in ['calories','protein_g','fat_g','carbs_g']}
    return {'date':diary.log_date,'entries':diary.entries,
            'totals':totals,'targets':targets,'remaining':remaining,'pct_complete':pct}


def print_daily_summary(s: dict):
    t, g, r, p = s['totals'], s['targets'], s['remaining'], s['pct_complete']
    print(f'ДНЕВНО РЕЗЮМЕ - {s["date"]}')
    print('=' * 60)
    for label, key in [('Калории','calories'),('Протеин г','protein_g'),
                        ('Мазнини г','fat_g'),('Въглехидрати г','carbs_g')]:
        bar = '█'*min(int(p[key]/10),10) + '░'*(10-min(int(p[key]/10),10))
        ov  = ' ⚠️ НАДВИШЕНО' if r[key] < 0 else ''
        print(f'  {label:<16} {t[key]:>7.0f} / {g[key]:>7}  {bar}  {p[key]:>5.1f}%{ov}')
    print()
    if r['protein_g'] > 20:
        print(f'  💡 Остават {r["protein_g"]:.0f}г протеин - добави ~{r["protein_g"]/25:.0f} порции пиле/яйца')
    if r['calories'] > 0:
        print(f'  📊 Остават {r["calories"]:.0f} ккал за деня')
    if len(s['entries']) > 0:
        low_conf = sum(1 for e in s['entries'] if e.get('confidence') == 'low')
        if low_conf:
            print(f'  🔴 {low_conf} хранения с LLM оценка (ниска точност) — провери ги')

print('DailyLog and tracking functions defined')

DailyLog and tracking functions defined


## 6. Meal Suggestion

In [19]:
@retry(wait=wait_exponential(min=1, max=20), stop=stop_after_attempt(3))
def suggest_meal(remaining: dict, preferences: str = '', restrictions: str = '') -> str:
    prompt = f"""Ти си спациалист по диетология и здравословно хранене. Оставащи макроси за деня:
- Калории: {remaining['calories']:.0f} ккал
- Протеин: {remaining['protein_g']:.0f} г
- Мазнини: {remaining['fat_g']:.0f} г  
- Въглехидрати: {remaining['carbs_g']:.0f} г
Предпочитания: {preferences or 'няма'}
Ограничения: {restrictions or 'няма'}

Предложи 2-3 конкретни хранения с точни количества в грамове спрямо посочените макроси, калории, предпочитания и ограничения.
Отговаряй на БЪЛГАРСКИ, кратко и практично."""

    r = openai_client.chat.completions.create(
        model=LLM_MODEL, messages=[{'role':'user','content':prompt}],
        temperature=0.5, max_tokens=350,
    )
    return r.choices[0].message.content

print('Meal suggestion defined')

Meal suggestion defined


## 7. Full Day Simulation

In [20]:
USER_ID = 'test_001'
TARGETS = {'calories': 2405, 'protein_g': 176, 'fat_g': 70, 'carbs_g': 268}

DailyLog(USER_ID).clear()
print('=== СИМУЛАЦИЯ НА ПЪЛЕН ДЕН ===\n')

meals = [
    ('ЗАКУСКА', '4 яйца бъркани с масло и 100г овесени ядки'),
    ('ОБЯД', '200г пилешки гърди на скара и 150г печен ориз на фурна със зеленчуци'),
    ('СЛЕДТРЕН. ШЕЙК', '40г суроватъчен протеин прах с вода'),
]

for meal_name, food_text in meals:
    print(f'--- {meal_name} ---')
    result = log_food(USER_ID, food_text)
    for e in result['entries_added']:
        icon = '🟢' if e['confidence']=='high' else '🟡' if e['confidence']=='medium' else '🔴'
        print(f'  {icon} {e["food_name"]:<28} {e["quantity_g"]:>5}г | '
              f'{e["calories"]:>5.0f} ккал | P:{e["protein_g"]:>4.1f}г')
    print(f'  Общо до момента: {result["daily_totals"]["calories"]:.0f} ккал')
    print()

summary = get_daily_summary(USER_ID, TARGETS)
print_daily_summary(summary)

print()
print('--- ПРЕДЛОЖЕНИЕ ЗА ВЕЧЕРЯ ---')
print(suggest_meal(summary['remaining'], 'обичам риба', 'няма'))

=== СИМУЛАЦИЯ НА ПЪЛЕН ДЕН ===

--- ЗАКУСКА ---
  Общо до момента: 0 ккал

--- ОБЯД ---
  Общо до момента: 0 ккал

--- СЛЕДТРЕН. ШЕЙК ---
  Общо до момента: 0 ккал

ДНЕВНО РЕЗЮМЕ - 2026-05-10
  Калории                0 /    2405  ░░░░░░░░░░    0.0%
  Протеин г              0 /     176  ░░░░░░░░░░    0.0%
  Мазнини г              0 /      70  ░░░░░░░░░░    0.0%
  Въглехидрати г         0 /     268  ░░░░░░░░░░    0.0%

  💡 Остават 176г протеин - добави ~7 порции пиле/яйца
  📊 Остават 2405 ккал за деня

--- ПРЕДЛОЖЕНИЕ ЗА ВЕЧЕРЯ ---
Разбира се! Ето три предложения за хранения, които отговарят на посочените макроси:

### Хранене 1: Риба със зеленчуци и киноа
- **Печена сьомга**: 200 г (около 400 ккал, 50 г протеин, 22 г мазнини)
- **Киноа**: 150 г (сварена) (около 180 ккал, 6 г протеин, 3 г мазнини, 30 г въглехидрати)
- **Смесени зеленчуци (броколи, моркови, чушки)**: 200 г (около 70 ккал, 5 г протеин, 0 г мазнини, 15 г въглехидрати)
  
**Общо**: 650 ккал, 61 г протеин, 25 г мазнини, 45 г 

## 8. Summary

In [21]:
print('=' * 55)
print('  NOTEBOOK 09 — COMPLETE')
print('=' * 55)
print()
print('  Nutrition data sources:')
print('  🟢 USDA FoodData Central  — gov. verified, ~300K foods')
print('  🔴 GPT-4o-mini fallback   — for foods not in USDA')
print()
print('  Features:')
print('  - parse_food_entry()    : BG text → nutrition data')
print('  - log_food()            : add entries to daily log')
print('  - get_daily_summary()   : totals vs targets + %')
print('  - print_daily_summary() : formatted display')
print('  - suggest_meal()        : meal ideas for remaining macros')


  NOTEBOOK 09 — COMPLETE

  Nutrition data sources:
  🟢 USDA FoodData Central  — gov. verified, ~300K foods
  🔴 GPT-4o-mini fallback   — for foods not in USDA

  Features:
  - parse_food_entry()    : BG text → nutrition data
  - log_food()            : add entries to daily log
  - get_daily_summary()   : totals vs targets + %
  - print_daily_summary() : formatted display
  - suggest_meal()        : meal ideas for remaining macros
